In [1]:
#Purpose
#analysing rising sea level trends in the Pacific and assessing the projected impact on low-lying nations

#Step 0 Importing Libraries
library(tidyverse)
library(ggplot2)
library(reshape2)

Warning message:
"package 'ggplot2' was built under R version 4.5.3"
── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'reshape2' was built under R version 4.5.3"

Attaching package: 'reshape2'


The following object is masked from 'package:tidyr':

    smiths




In [2]:
#Step 0 Importing dataset and inspection
main_df <- read.csv("USStationsLinearSeaLevelTrends.csv")

In [3]:
glimpse(main_df) #Transposed view of entire dataset, running horizontal instead of vertical

Rows: 149
Columns: 12
$ Station.ID               <int> 1611400, 1612340, 1612480, 1615680, 1617433, …
$ Station.Name             <chr> "Nawiliwili, HI", "Honolulu, HI", "Mokuoloe, …
$ First.Year               <int> 1955, 1905, 1957, 1947, 1988, 1927, 1947, 199…
$ Last.Year                <int> 2025, 2025, 2025, 2025, 2025, 2025, 2025, 202…
$ Year.Range               <int> 71, 121, 69, 79, 38, 99, 79, 33, 17, 80, 76, …
$ X..Complete              <int> 100, 100, 84, 96, 92, 85, 96, 96, 95, 99, 94,…
$ MSL.Trends..mm.yr.       <dbl> 1.90, 1.57, 1.79, 2.30, 3.81, 3.15, 1.58, 5.3…
$ X....95..CI..mm.yr.      <dbl> 0.37, 0.19, 0.47, 0.36, 0.96, 0.27, 0.36, 2.8…
$ MSL.Trend..ft.century.   <dbl> 0.62, 0.51, 0.59, 0.75, 1.25, 1.03, 0.52, 1.7…
$ X....95..CI..ft.century. <dbl> 0.12, 0.06, 0.16, 0.12, 0.31, 0.09, 0.12, 0.9…
$ Latitude                 <dbl> 21.9544, 21.3033, 21.4331, 20.8949, 20.0366, …
$ Longitude                <dbl> -159.3561, -157.8645, -157.7900, -156.4690, -…


In [4]:
head(main_df) #irst 5 rows of the Relative Sea Level Trends for Honolulu, Hawaii

,Station.ID,Station.Name,First.Year,Last.Year,Year.Range,X..Complete,MSL.Trends..mm.yr.,X....95..CI..mm.yr.,MSL.Trend..ft.century.,X....95..CI..ft.century.,Latitude,Longitude
,<int>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1611400,"Nawiliwili, HI",1955,2025,71,100,1.90,0.37,0.62,0.12,21.9544,-159.3561
2,1612340,"Honolulu, HI",1905,2025,121,100,1.57,0.19,0.51,0.06,21.3033,-157.8645
3,1612480,"Mokuoloe, HI",1957,2025,69,84,1.79,0.47,0.59,0.16,21.4331,-157.7900
4,1615680,"Kahului, HI",1947,2025,79,96,2.30,0.36,0.75,0.12,20.8949,-156.4690
5,1617433,"Kawaihae, HI",1988,2025,38,92,3.81,0.96,1.25,0.31,20.0366,-155.8294
6,1617760,"Hilo, HI",1927,2025,99,85,3.15,0.27,1.03,0.09,19.7303,-155.0556


In [ ]:
#The dataset I've used contains 149 US tide gauge stations
#Columns include: Station.ID, Station.Name, First.Year, Last.Year, Year.Range, X..Complete, 
#MSL.Trends..mm.yr., X....95..CI..mm.yr.
#MSL.Trend.ft.century.
#X....95..CI..ft.century.
#Latitude
#Longitude

In [6]:
#Step 1: Transforming into wide format
#Step 1.1 - Building the Mean Sea Level Matrix
unique(main_df$Station.Name) #Checking all the unique region names in the dataset

[1] "Nawiliwili, HI"                   "Honolulu, HI"                    
  [3] "Mokuoloe, HI"                     "Kahului, HI"                     
  [5] "Kawaihae, HI"                     "Hilo, HI"                        
  [7] "Midway Atoll, "                   "Apra Harbor, Guam, "             
  [9] "Pago Pago, American Samoa, "      "Kwajalein, Marshall Islands, "   
 [11] "Wake Island, Pacific Ocean, "     "Biological Station, Bermuda, "   
 [13] "St. Georges, Bermuda, "           "Eastport, ME"                    
 [15] "Cutler, ME"                       "Bar Harbor, ME"                  
 [17] "Portland, ME"                     "Seavey Island, ME"               
 [19] "Fort Point, NH"                   "Boston, MA"                      
 [21] "Woods Hole, MA"                   "Nantucket Island, MA"            
 [23] "Newport, RI"                      "Providence, RI"                  
 [25] "New London, CT"                   "Bridgeport, CT"                  
 [27] "Montauk, NY"                      "Kings Point, NY"                 
 [29] "The Battery, NY"                  "Bergen Point, NY"                
 [31] "Sandy Hook, NJ"                   "Atlantic City, NJ"               
 [33] "Cape May, NJ"                     "Philadelphia, PA"                
 [35] "Reedy Point, DE"                  "Lewes, DE"                       
 [37] "Ocean City, MD"                   "Cambridge, MD"                   
 [39] "Tolchester Beach, MD"             "Chesapeake City, MD"             
 [41] "Baltimore, MD"                    "Annapolis, MD"                   
 [43] "Solomons Island, MD"              "Washington, DC"                  
 [45] "Wachapreague, VA"                 "Kiptopeke, VA"                   
 [47] "Dahlgren, VA"                     "Lewisetta, VA"                   
 [49] "Windmill Point, VA"               "Yorktown, VA"                    
 [51] "Sewells Point, VA"                "Chesapeake Bay Bridge Tunnel, VA"
 [53] "Chesapeake Channel, VA"           "Duck, NC"                        
 [55] "Oregon Inlet Marina, NC"          "Beaufort, NC"                    
 [57] "Wilmington, NC"                   "Springmaid Pier, SC"             
 [59] "Charleston, SC"                   "Fort Pulaski, GA"                
 [61] "Fernandina Beach, FL"             "Mayport, FL"                     
 [63] "Port Canaveral, FL"               "Lake Worth Pier, FL"             
 [65] "Virginia Key, FL"                 "Vaca Key, FL"                    
 [67] "Key West, FL"                     "Naples, FL"                      
 [69] "Fort Myers, FL"                   "Port Manatee, FL"                
 [71] "St. Petersburg, FL"               "Old Port Tampa, FL"              
 [73] "East Bay, FL"                     "Clearwater Beach, FL"            
 [75] "Cedar Key, FL"                    "Apalachicola, FL"                
 [77] "Panama City, FL"                  "Panama City Beach, FL"           
 [79] "Pensacola, FL"                    "Dauphin Island, AL"              
 [81] "Mobile State Docks, AL"           "Bay Waveland, MS"                
 [83] "Grand Isle, LA"                   "New Canal, LA"                   
 [85] "Sabine Pass, TX"                  "Morgans Point, TX"               
 [87] "Texas Point, TX"                  "Eagle Point, TX"                 
 [89] "Galveston Bay Entrance, TX"       "Galveston Pier 21, TX"           
 [91] "Galveston Pleasure Pier, TX"      "Freeport, TX"                    
 [93] "Freeport Harbor, TX"              "Rockport, TX"                    
 [95] "Packery Channel, TX"              "Corpus Christi, TX"              
 [97] "Port Mansfield, TX"               "South Padre Island, TX"          
 [99] "Port Isabel, TX"                  "San Diego, CA"                   
[101] "La Jolla, CA"                     "Los Angeles, CA"                 
[103] "Santa Monica, CA"                 "Santa Barbara, CA"               
[105] "Port San Luis, CA"                "Monterey

In [8]:
main_df$MSL.Trends..mm.yr. #Checking the Mean Sea Level trends in mm per year for each station
#This column asks how quickly the sea is threatening each of the station's location

[1]   1.90   1.57   1.79   2.30   3.81   3.15   1.58   5.37  15.48   2.12
 [11]   2.15   2.21   2.19   2.33   2.34   2.48   1.97   2.21   2.04   2.97
 [21]   3.15   4.14   2.98   2.64   2.94   3.43   3.60   2.74   2.95   4.72
 [31]   4.28   4.25   5.10   3.15   3.94   3.78   5.25   4.05   4.31   4.47
 [41]   3.31   3.87   4.11   3.54   5.68   4.02   5.70   5.93   6.47   4.84
 [51]   4.84   5.92   6.15   4.94   5.50   3.62   2.80   3.36   3.51   3.67
 [61]   2.31   2.92   6.41   4.12   3.20   4.19   2.64   3.35   3.62   5.46
 [71]   3.13   6.04   5.50   4.36   2.39   3.05   3.08   4.86   2.72   4.37
 [81]   4.62   4.59   9.13   5.99   6.16   3.80   5.76  12.19   6.59   6.63
 [91]   6.62   4.21   3.51   6.04   6.13   5.48   3.73   4.09   4.35   2.22
[101]   2.02   1.07   1.52   1.05   0.99   1.77   1.99   2.74   0.94   3.35
[111]   2.15   1.96   1.15   5.08  -0.76  -0.05   1.12   1.75   2.44  -0.15
[121]   0.45  -1.70   0.46   1.82   2.09  -0.04   1.19  -0.36  -2.46 -13.51
[131] -18.13 -15.40  -0.54  -8.77  -3.08  -9.51 -10.12  -0.69  -9.53   2.04
[141]  -2.30  -4.18   4.02   4.15   4.28   2.71   2.21   2.16   2.06

In [12]:
sea_matrix <- matrix(
    main_df$MSL.Trends..mm.yr, #Pulls all the 149 values from the MSL.Trends..mm.yr column
    nrow = 149 #Total rows of the dataset
    )

rownames(sea_matrix) <- main_df$Station.Name #Assigning station names as row labels
colnames(sea_matrix) <- "MSL_Trend_mm_yr"
         
print(sea_matrix) #Viewing the matrix

#The reason why I chose to do only station.name and MSL_Trend_mm_yr is because I want to see the correlation betwen each station's MSL.

                                 MSL_Trend_mm_yr
Nawiliwili, HI                              1.90
Honolulu, HI                                1.57
Mokuoloe, HI                                1.79
Kahului, HI                                 2.30
Kawaihae, HI                                3.81
Hilo, HI                                    3.15
Midway Atoll,                               1.58
Apra Harbor, Guam,                          5.37
Pago Pago, American Samoa,                 15.48
Kwajalein, Marshall Islands,                2.12
Wake Island, Pacific Ocean,                 2.15
Biological Station, Bermuda,                2.21
St. Georges, Bermuda,                       2.19
Eastport, ME                                2.33
Cutler, ME                                  2.34
Bar Harbor, ME                              2.48
Portland, ME                                1.97
Seavey Island, ME                           2.21
Fort Point, NH                              2.04
Boston, MA          

In [13]:
#Step 1.2 Transposing the Matrix into Wide Format
wide_matrix <- t(sea_matrix) #Station names become columns and MSL_Trend_mm_yr become rows. 

print(wide_matrix) #The wide format matrix contains 149 columns (one per station) which makes it difficult to display in full, however the structure is correct with stations as columns and MSL_Trend_mm_yr as the single row.

                Nawiliwili, HI Honolulu, HI Mokuoloe, HI Kahului, HI
MSL_Trend_mm_yr            1.9         1.57         1.79         2.3
                Kawaihae, HI Hilo, HI Midway Atoll,  Apra Harbor, Guam, 
MSL_Trend_mm_yr         3.81     3.15           1.58                5.37
                Pago Pago, American Samoa,  Kwajalein, Marshall Islands, 
MSL_Trend_mm_yr                       15.48                          2.12
                Wake Island, Pacific Ocean,  Biological Station, Bermuda, 
MSL_Trend_mm_yr                         2.15                          2.21
                St. Georges, Bermuda,  Eastport, ME Cutler, ME Bar Harbor, ME
MSL_Trend_mm_yr                   2.19         2.33       2.34           2.48
                Portland, ME Seavey Island, ME Fort Point, NH Boston, MA
MSL_Trend_mm_yr         1.97              2.21           2.04       2.97
                Woods Hole, MA Nantucket Island, MA Newport, RI Providence, RI
MSL_Trend_mm_yr           3.15       

In [15]:
wide_matrix[, 1:6] #Previewing first 6 columns of the wide matrix to verify correct values 

Nawiliwili, HI   Honolulu, HI   Mokuoloe, HI    Kahului, HI   Kawaihae, HI 
          1.90           1.57           1.79           2.30           3.81 
      Hilo, HI 
          3.15

In [17]:
# Cross check against original data
head(main_df[, c("Station.Name", "MSL.Trends..mm.yr.")], 6)

,Station.Name,MSL.Trends..mm.yr.
,<chr>,<dbl>
1,"Nawiliwili, HI",1.90
2,"Honolulu, HI",1.57
3,"Mokuoloe, HI",1.79
4,"Kahului, HI",2.30
5,"Kawaihae, HI",3.81
6,"Hilo, HI",3.15


In [18]:
#Step 1.3 Scaling the wide_matrix
scaled_matrix <- scale(wide_matrix) #standardises all values to have a mean of 0 and standard deviation of 1. Without doing this first, high values of MSL.Trends..mm.yr will distort the correlation

print(scaled_matrix)
                       

                Nawiliwili, HI Honolulu, HI Mokuoloe, HI Kahului, HI
MSL_Trend_mm_yr            NaN          NaN          NaN         NaN
                Kawaihae, HI Hilo, HI Midway Atoll,  Apra Harbor, Guam, 
MSL_Trend_mm_yr          NaN      NaN            NaN                 NaN
                Pago Pago, American Samoa,  Kwajalein, Marshall Islands, 
MSL_Trend_mm_yr                         NaN                           NaN
                Wake Island, Pacific Ocean,  Biological Station, Bermuda, 
MSL_Trend_mm_yr                          NaN                           NaN
                St. Georges, Bermuda,  Eastport, ME Cutler, ME Bar Harbor, ME
MSL_Trend_mm_yr                    NaN          NaN        NaN            NaN
                Portland, ME Seavey Island, ME Fort Point, NH Boston, MA
MSL_Trend_mm_yr          NaN               NaN            NaN        NaN
                Woods Hole, MA Nantucket Island, MA Newport, RI Providence, RI
MSL_Trend_mm_yr            NaN       